# 02 · Control statistics audit

Three questions about the conditioning signals stored in the tensors.

**Part A — what range of controls did the model see?** Distribution of `tension` and `combat_score` per source, measured on every tensor. Phase 2 uses these percentiles to choose in-distribution control levels for generation.

**Part B — which tension normalisation was used for each block of tensors?** `configs/norm_stats.json` has had three versions of `tension_rms` (the tokenizer notebook re-estimated it from randomly subsampled RMS frames in every run), and the tensors were encoded in several runs. On unclipped frames the stored tension is a linear function of the frame RMS, so recomputing the RMS from the original audio with the tokenizer pipeline and fitting a line recovers the `p_low` / `p_high` that were used. The check also recognises the early Colab tokenizer (`notebooks/archive/colab_ost_features_early_tokenizer.ipynb`: floor frame size, per-track min-max tension).

**Part C — was the combat score computed with the same pipeline?** The notebook that encoded the OST tensors is not in the repository. Because the combat window is causal, recomputing the first seconds of a track with `src/control_features.py` (a verbatim copy of the Slakh tokenizer functions) must give the stored values. Part C also measures the raw window-level features against the normalisation ranges, which were computed on whole-track features (`csv/`).

**Kaggle inputs to attach:** `slakh-tensors-final`, `ost-tensors-sky-w3`, `slakh2100` (alonhaviv), `skyrim-ost`, `witcher3-ost`. Internet must be on (the notebook clones the repository and installs `encodec`).

In [ ]:
# ── Repository bootstrap: make src/ importable on Kaggle and locally ──
# On Kaggle the repository is cloned into /kaggle/working (Internet must be
# enabled in the notebook settings). Locally, set SECUNDA_REPO_DIR to the
# repository root and nothing is cloned.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DanBracs7/Adaptive-Secunda-Synth.git"
REPO_BRANCH = "revision"  # switch to "main" after merging the revision
REPO_DIR = Path(os.environ.get("SECUNDA_REPO_DIR", "/kaggle/working/Adaptive-Secunda-Synth"))

if not (REPO_DIR / "src").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "src"))

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit or "unknown (not a git checkout)")

In [ ]:
from paths import get_path, running_on_kaggle, source_roots_for_split

if running_on_kaggle():
    # Same codec package used by the tokenizer notebook (convert_audio).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "encodec==0.1.1"], check=True)

In [ ]:
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from control_features import (
    COMBAT_FEATURES,
    KNOWN_TENSION_RMS_STATS,
    align_causal_to_frames,
    closest_known_tension_stats,
    compute_causal_combat_score_track,
    fit_tension_stats,
    frame_rms,
    frame_rms_legacy,
    load_norm_stats,
    load_waveform_like_tokenizer,
)

OUTPUT_DIR = get_path("output_dir") / "control_statistics_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SLAKH_AUDIO_ROOT = Path(os.environ.get(
    "SECUNDA_SLAKH_AUDIO",
    "/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux",
))
OST_AUDIO_ROOTS = {
    "skyrim": Path(os.environ.get(
        "SECUNDA_SKYRIM_AUDIO", "/kaggle/input/datasets/danielebracoloni/skyrim-ost",
    )),
    "witcher3": Path(os.environ.get(
        "SECUNDA_WITCHER3_AUDIO", "/kaggle/input/datasets/danielebracoloni/witcher3-ost",
    )),
}

# Part B checks a deterministic subset of tensors.
SLAKH_TRAIN_STRIDE = 25   # train_000, train_025, ...
SLAKH_TEST_STRIDE = 10    # test_000, test_010, ...
OST_STRIDE = 4            # every 4th OST tensor, in file order

# A fit is assigned to a known version when every statistic is within this
# distance. The closest known versions differ by about 3e-3 in p_high.
MATCH_TOLERANCE = 2e-4
# A larger residual means the audio or the pipeline differs from the run
# that produced the tensor.
RESIDUAL_TOLERANCE = 1e-4

# Part C recomputes the combat score of the first COMBAT_SECONDS of
# COMBAT_TRACKS_PER_BLOCK tracks per block (about 0.5 s of CPU per hop).
COMBAT_TRACKS_PER_BLOCK = 2
COMBAT_SECONDS = int(os.environ.get("SECUNDA_COMBAT_SECONDS", 60))
SAMPLE_RATE = 24_000
FRAME_RATE = 75

NORM_STATS = load_norm_stats(get_path("norm_stats"))
print("configs/norm_stats.json tension_rms:", NORM_STATS["tension_rms"])
for name, stats in KNOWN_TENSION_RMS_STATS.items():
    print(f"known {name}: {stats}")

In [ ]:
with open(get_path("manifest"), newline="", encoding="utf-8") as file:
    manifest_rows = [
        row for row in csv.DictReader(file)
        if row["split"] != "excluded"
    ]


def block_of(row):
    stem = Path(row["path"]).stem
    if row["source"] != "slakh":
        return row["source"]
    if stem.startswith("test_"):
        return "slakh test"
    index = int(stem.split("_")[-1])
    if index < 800:
        return "slakh 0000-0799"
    if index < 1070:
        return "slakh 0800-1069"
    return "slakh 1070-1394"


def group_of(row):
    if row["source"] == "slakh":
        return "Slakh test" if row["split"] == "test" else "Slakh train+val"
    return {"skyrim": "Skyrim", "witcher3": "Witcher 3"}[row["source"]]


def tensor_path_of(row):
    return source_roots_for_split(row["split"])[row["source"]] / Path(row["path"]).name


def load_tensor(path):
    # mmap avoids reading the token matrix when only the controls are needed.
    try:
        return torch.load(path, map_location="cpu", mmap=True)
    except RuntimeError:
        return torch.load(path, map_location="cpu")


print("Usable manifest rows:", len(manifest_rows))
print(pd.Series([block_of(row) for row in manifest_rows]).value_counts().sort_index().to_string())

## Part A — distribution of the controls seen in training

In [ ]:
controls = {}

for index, row in enumerate(manifest_rows, start=1):
    data = load_tensor(tensor_path_of(row))
    for key in (group_of(row), block_of(row)):
        bucket = controls.setdefault(key, {"tension": [], "combat_score": []})
        bucket["tension"].append(data["tension"].float().numpy().copy())
        bucket["combat_score"].append(data["combat_score"].float().numpy().copy())
    if index % 200 == 0:
        print(f"read {index}/{len(manifest_rows)} tensors")

controls = {
    key: {signal: np.concatenate(chunks) for signal, chunks in bucket.items()}
    for key, bucket in controls.items()
}
print("Done.")

In [ ]:
PERCENTILES = [5, 10, 25, 50, 75, 90, 95]
GROUP_ORDER = ["Slakh train+val", "Slakh test", "Skyrim", "Witcher 3"]
BLOCK_ORDER = ["slakh 0000-0799", "slakh 0800-1069", "slakh 1070-1394", "slakh test", "skyrim", "witcher3"]

distribution_rows = []
for key in GROUP_ORDER + BLOCK_ORDER:
    if key not in controls:
        continue
    tension = controls[key]["tension"]
    combat = controls[key]["combat_score"]
    for signal, values in (("tension", tension), ("combat_score", combat)):
        row = {"group": key, "signal": signal, "frames": len(values)}
        row.update({f"p{p}": float(np.percentile(values, p)) for p in PERCENTILES})
        row["frac_at_0"] = float(np.mean(values <= 0.0))
        row["frac_at_1"] = float(np.mean(values >= 1.0))
        row["pearson_tension_combat"] = float(np.corrcoef(tension, combat)[0, 1])
        distribution_rows.append(row)

distributions = pd.DataFrame(distribution_rows)
distributions.to_csv(OUTPUT_DIR / "control_distributions.csv", index=False)
print(distributions.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0.0, 1.0, 51)

for axis, signal in zip(axes, ("tension", "combat_score")):
    for group in GROUP_ORDER:
        if group in controls:
            axis.hist(controls[group][signal], bins=bins, density=True, histtype="step", linewidth=1.5, label=group)
    axis.set_title(f"{signal} per frame")
    axis.set_xlabel("value")
    axis.set_ylabel("density")
    axis.legend()

figure.tight_layout()
figure.savefig(OUTPUT_DIR / "control_distributions.png", dpi=150)
plt.show()

## Part B — tension statistics used for each block

In [ ]:
def normalized_name(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


slakh_audio = {}
if SLAKH_AUDIO_ROOT.exists():
    for split_dir in sorted(SLAKH_AUDIO_ROOT.iterdir()):
        if split_dir.is_dir():
            for mix_path in split_dir.glob("*/mix.flac"):
                slakh_audio[mix_path.parent.name] = mix_path
print("Slakh mixes indexed:", len(slakh_audio))

AUDIO_SUFFIXES = {".mp3", ".flac", ".wav", ".ogg"}
ost_audio = {}
for source, root in OST_AUDIO_ROOTS.items():
    files = [p for p in root.rglob("*") if p.suffix.lower() in AUDIO_SUFFIXES] if root.exists() else []
    ost_audio[source] = [(normalized_name(p.stem), p) for p in sorted(files)]
    print(f"{source} audio files indexed:", len(files))


def find_audio(row, track_id):
    if row["source"] == "slakh":
        return slakh_audio.get(track_id), "track_id lookup"
    candidates = ost_audio.get(row["source"], [])
    wanted = normalized_name(track_id)
    exact = [path for name, path in candidates if name == wanted]
    if len(exact) > 1:
        # Same title in several formats: prefer FLAC, as the Colab
        # notebook did (find_audio_files_deduped).
        flac = [path for path in exact if path.suffix.lower() == ".flac"]
        exact = flac if len(flac) == 1 else exact
    if len(exact) == 1:
        return exact[0], "exact title"
    partial = [path for name, path in candidates if wanted and wanted in name]
    if len(partial) == 1:
        return partial[0], "title substring"
    return None, f"{len(partial)} candidates"

In [ ]:
def selected_for_part_b(row, position_in_source):
    stem = Path(row["path"]).stem
    if row["source"] == "slakh":
        index = int(stem.split("_")[-1])
        stride = SLAKH_TEST_STRIDE if stem.startswith("test_") else SLAKH_TRAIN_STRIDE
        return index % stride == 0
    return position_in_source % OST_STRIDE == 0


def classify_tension(rms_frames, stored_tension):
    """
    Returns (status, fit). status is a known version ("v1", "v2", "v3"),
    "per-track min-max", "unknown statistics" or None when the RMS frames
    do not explain the stored tension (large residual or no fit).
    """
    fit = fit_tension_stats(rms_frames, stored_tension)
    if fit is None or fit["max_abs_residual"] > RESIDUAL_TOLERANCE:
        return None, fit
    version, distance = closest_known_tension_stats(fit)
    fit["closest_version"], fit["distance"] = version, distance
    if distance <= MATCH_TOLERANCE:
        return version, fit
    rms = torch.as_tensor(rms_frames)
    if (abs(fit["p_low"] - float(rms.min())) <= MATCH_TOLERANCE
            and abs(fit["p_high"] - float(rms.max())) <= MATCH_TOLERANCE):
        return "per-track min-max", fit
    return "unknown statistics", fit


positions = {}
fit_rows = []
audio_paths = {}

for row in sorted(manifest_rows, key=lambda r: (r["source"], r["path"])):
    position = positions.get(row["source"], 0)
    positions[row["source"]] = position + 1
    if not selected_for_part_b(row, position):
        continue

    data = load_tensor(tensor_path_of(row))
    track_id = str(data["track_id"])
    record = {"block": block_of(row), "file": Path(row["path"]).name, "track_id": track_id}

    audio_path, how = find_audio(row, track_id)
    record["audio_match"] = how
    if audio_path is None:
        record["status"] = "audio not found"
        fit_rows.append(record)
        continue

    wav_mono = load_waveform_like_tokenizer(audio_path)
    num_frames = int(data["tokens"].shape[-1])

    # Slakh tokenizer frames first, then the early Colab frames.
    status, fit = classify_tension(frame_rms(wav_mono, num_frames), data["tension"])
    record["frames"] = "tokenizer"
    if status is None:
        legacy_status, legacy_fit = classify_tension(
            frame_rms_legacy(wav_mono, num_frames), data["tension"]
        )
        if legacy_status is not None:
            status, fit = legacy_status, legacy_fit
            record["frames"] = "early Colab (legacy)"

    if fit is None:
        record["status"] = "too few unclipped frames"
        fit_rows.append(record)
        continue

    record.update(fit)
    record["status"] = status if status is not None else "pipeline mismatch"
    audio_paths[(record["block"], record["file"])] = (audio_path, row)

    fit_rows.append(record)
    print(f"{record['block']:16s} {record['file']:16s} -> {record['status']:20s} "
          f"frames={record['frames']:10s} p_low={fit['p_low']:.6f} p_high={fit['p_high']:.6f} "
          f"residual={fit['max_abs_residual']:.1e}")

fits = pd.DataFrame(fit_rows)
fits.to_csv(OUTPUT_DIR / "tension_stats_per_tensor.csv", index=False)

In [ ]:
summary = (
    fits.groupby(["block", "status"]).size().unstack(fill_value=0)
    .reindex([b for b in BLOCK_ORDER if b in set(fits["block"])])
)
summary.to_csv(OUTPUT_DIR / "tension_stats_per_block.csv")
print(summary.to_string())

# Largest possible difference in normalised tension between the versions.
rms_grid = np.linspace(0.0, 0.3, 30001)
names = list(KNOWN_TENSION_RMS_STATS)
for i, name_a in enumerate(names):
    for name_b in names[i + 1:]:
        a, b = KNOWN_TENSION_RMS_STATS[name_a], KNOWN_TENSION_RMS_STATS[name_b]
        tension_a = np.clip((rms_grid - a["p_low"]) / (a["p_high"] - a["p_low"] + 1e-8), 0, 1)
        tension_b = np.clip((rms_grid - b["p_low"]) / (b["p_high"] - b["p_low"] + 1e-8), 0, 1)
        print(f"max |tension({name_a}) - tension({name_b})| = {np.abs(tension_a - tension_b).max():.4f}")

## Part C — combat score recomputed from the audio

In [ ]:
import time

COMBAT_MATCH_TOLERANCE = 1e-4   # same pipeline and library versions
COMBAT_CLOSE_TOLERANCE = 2e-2   # same pipeline, different librosa version

combat_rows = []
component_chunks = {}

for block in BLOCK_ORDER:
    keys = [key for key in audio_paths if key[0] == block][:COMBAT_TRACKS_PER_BLOCK]
    for key in keys:
        audio_path, row = audio_paths[key]
        started = time.time()

        data = load_tensor(tensor_path_of(row))
        stored_combat = data["combat_score"].float().numpy()

        # The window is causal: the first K hops only depend on the first K
        # seconds of the (whole-track peak-normalised) waveform.
        y = load_waveform_like_tokenizer(audio_path).numpy()
        if len(y) > COMBAT_SECONDS * SAMPLE_RATE:
            y = y[:COMBAT_SECONDS * SAMPLE_RATE]
            compared_frames = COMBAT_SECONDS * FRAME_RATE
        else:
            compared_frames = len(stored_combat)

        scores, hop_sec, components = compute_causal_combat_score_track(
            y, SAMPLE_RATE, NORM_STATS, return_components=True
        )
        recomputed = align_causal_to_frames(scores, hop_sec, FRAME_RATE, compared_frames)
        difference = np.abs(recomputed - stored_combat[:compared_frames])

        if difference.max() <= COMBAT_MATCH_TOLERANCE:
            status = "match"
        elif difference.max() <= COMBAT_CLOSE_TOLERANCE:
            status = "close"
        else:
            status = "different"

        component_chunks.setdefault(block, []).append(components)
        combat_rows.append({
            "block": block,
            "file": key[1],
            "compared_seconds": round(compared_frames / FRAME_RATE, 1),
            "max_abs_difference": float(difference.max()),
            "mean_abs_difference": float(difference.mean()),
            "status": status,
        })
        print(f"{block:16s} {key[1]:16s} -> {status:9s} max|diff|={difference.max():.2e} "
              f"({time.time() - started:.0f} s)")

combat_check = pd.DataFrame(combat_rows)
combat_check.to_csv(OUTPUT_DIR / "combat_recomputation.csv", index=False)
print()
print(combat_check.to_string(index=False))

In [ ]:
# Raw window-level features against the normalisation ranges, which were
# computed on whole-track features (csv/). A component that is clipped at
# 0 or 1 most of the time contributes little information to the score.
range_rows = []

for block, chunks in component_chunks.items():
    values = np.concatenate(chunks)
    values = values[~np.isnan(values).any(axis=1)]
    for index, feature in enumerate(COMBAT_FEATURES):
        column = values[:, index]
        p_low = NORM_STATS[feature]["p_low"]
        p_high = NORM_STATS[feature]["p_high"]
        range_rows.append({
            "block": block,
            "feature": feature,
            "windows": len(column),
            "window_p5": float(np.percentile(column, 5)),
            "window_p50": float(np.percentile(column, 50)),
            "window_p95": float(np.percentile(column, 95)),
            "norm_p_low": p_low,
            "norm_p_high": p_high,
            "frac_clipped_at_0": float(np.mean(column <= p_low)),
            "frac_clipped_at_1": float(np.mean(column >= p_high)),
        })

feature_ranges = pd.DataFrame(range_rows)
feature_ranges.to_csv(OUTPUT_DIR / "combat_feature_ranges.csv", index=False)
print(feature_ranges.to_string(index=False, float_format=lambda value: f"{value:.4g}"))

### How to read the result

**Part B**
- One version for every block: the tension normalisation is consistent; `configs/norm_stats.json` must contain that version.
- Different versions across blocks: the model saw tension curves normalised with slightly different statistics (the maximum difference is printed above). The report states it, and controls measured on generated audio use the version of the majority of the training frames.
- `per-track min-max` with `early Colab (legacy)` frames: the tensor comes from the early Colab tokenizer, whose tension is not comparable across tracks.
- `pipeline mismatch`: the stored tension is not explained by either pipeline (different audio file or different processing).

**Part C**
- `match`: same combat pipeline and library versions.
- `close`: same pipeline; the small difference comes from a different librosa version on Kaggle.
- `different`: a different combat computation; Phase 2 must not assume the stored combat_score of that block follows `src/control_features.py`.
- `frac_clipped_*`: how often each feature saturates at window level, because the normalisation ranges come from whole-track features.